In [1]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3.5',
    dtype="auto",
    device_map="cuda",
    trust_remote_code=True,
)

model.eval()

/home/coene/.local/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /home/coene/.local/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [2]:
query = "Which predictor does the model depend on most heavily to make its predictions?"
docs = [
    "Machine learning is emerging as a valuable tool for convection-related applications such as post-processing numerical weather prediction output, improving understanding of convective storm climatology and potentially improving existing convective parameterization schemes. In a rapidly developing field, it is vital to assess the strengths and limitations of machine learning approaches across different applications. Here, a probabilistic model is developed using a convective cell dataset as ground truth and predictors primarily from ERA5. The model's ability to reproduce the convective cell climatology at different regions relative to cold fronts (i.e. post-frontal and pre-frontal) is assessed during the warm-season in Germany. The optimal number of features (predictors) is selected using a feature elimination strategy. Overall, the optimised model exhibits high skill in reproducing the spatial and temporal cell frequency at different regions relative to the front. While the highest cell frequency is correctly identified near the surface front, the model underestimates the actual cell count in this region. Feature importance analysis shows that the model depends most heavily on CAPE to make its predictions. Additionally, the time of day predictor is key for accurately capturing the diurnal cycle of convective cells on both sides of the cold front. The study highlights both the advantages and the limitations of data-driven models, offering valuable insights for future data-driven climate and weather prediction models."
]

In [3]:
scores = model.rerank(query, docs)
for score in scores:
    print(f"Index: {score['index']}")
    print(f"Score: {score['relevance_score']:.4f}")
    print(f"Document: {score['document'][:100]}...")
    print("-" * 100)

Index: 0
Score: 0.5767
Document: Machine learning is emerging as a valuable tool for convection-related applications such as post-pro...
----------------------------------------------------------------------------------------------------


In [25]:
import pandas as pd

df_questions_generals = pd.read_csv("../data/3_gold/questions_v2_rrf_union.csv")
df_questions_generals.describe()

,Unnamed: 0,doc_id_1,doc_id_2,doc_id_3,doc_id_4,doc_id_5,qwen_res,gemma_res,rrf_res,rrf_mrr,union_res
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000
mean,249.500000,4032.966000,4146.206000,4156.740000,4161.346000,4263.376000,0.361600,0.328000,0.36080,0.384082,0.424800
std,144.481833,2442.211788,2414.149222,2461.061102,2433.835404,2392.740464,0.251897,0.263048,0.25992,0.380427,0.268643
min,0.000000,3.000000,6.000000,0.000000,10.000000,8.000000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,124.750000,2048.750000,2033.250000,2028.000000,2114.500000,2146.250000,0.200000,0.200000,0.20000,0.076900,0.200000
50%,249.500000,3932.000000,4247.500000,4409.500000,4235.000000,4436.000000,0.400000,0.200000,0.40000,0.250000,0.400000
75%,374.250000,6177.000000,6226.750000,6105.000000,6215.500000,6322.750000,0.600000,0.600000,0.60000,0.500000,0.600000
max,499.000000,8433.000000,8404.000000,8368.000000,8427.000000,8440.000000,1.000000,1.000000,1.00000,1.000000,1.000000


In [ ]:
N = 10
for index, row in df_questions_generals.sample(n=N).iterrows():
    print(f"**** {index} ****")
    query = row['question']
    docs  = [
        row[f'abstract_{i}'] for i in range(1,6) 
    ]
    scores = model.rerank(query, docs)
    for score in scores:
        print(f"Index: {score['index']}")
        print(f"Score: {score['relevance_score']:.4f}")
        print(f"Document: {score['document'][:100]}...")
        print("-" * 100)
    print("*"*120
    )
    input("Enter para continuar")

**** 450 ****
Index: 4
Score: 0.1246
Document: Mesoscale eddies dominate the spatiotemporal multiscale variability of the ocean, and their impact o...
----------------------------------------------------------------------------------------------------
Index: 3
Score: 0.1156
Document: The emergence of oscillations in models of the El-Niño effect is of utmost relevance. Here we invest...
----------------------------------------------------------------------------------------------------
Index: 0
Score: 0.0449
Document: We study deep water ocean wind-driven waves in strait, with wind directed orthogonally to the shore,...
----------------------------------------------------------------------------------------------------
Index: 2
Score: -0.0200
Document: McDougall (1989) proved that neutral surfaces possess an exact geostrophic streamfunction, but its f...
----------------------------------------------------------------------------------------------------
Index: 1
Score: -0.1009
Document:

**** 247 ****
Index: 4
Score: 0.1155
Document:   This work is devoted to the study of solar light that experiences single and multiple scattering i...
----------------------------------------------------------------------------------------------------
Index: 3
Score: 0.0875
Document: Several light pollution indicators are commonly used to monitor the effects of the transition from o...
----------------------------------------------------------------------------------------------------
Index: 1
Score: 0.0650
Document: We investigate analytically and numerically correlation functions of the phase of light waves that p...
----------------------------------------------------------------------------------------------------
Index: 0
Score: -0.0736
Document:   The effect of a random phase diffuser on fluctuations of laser light (scintillations) is studied. ...
----------------------------------------------------------------------------------------------------
Index: 2
Score: -0.1105
Document:

In [26]:
# Inicializar las columnas en el DataFrame si no existen
if "max_score" not in df_questions_generals.columns:
    df_questions_generals["max_score"] = None
if "min_score" not in df_questions_generals.columns:
    df_questions_generals["min_score"] = None

In [27]:
for index, row in df_questions_generals.iterrows():
    print(f"**** {index} ****")
    query = row["question"]
    docs = [row[f"abstract_{i}"] for i in range(1, 6)]

    scores = model.rerank(query, docs)

    # Extraer los puntajes
    lista_scores = [s["relevance_score"] for s in scores]

    # Guardar los valores directamente en la fila del DataFrame usando .at
    df_questions_generals.at[index, "max_score"] = max(lista_scores)
    df_questions_generals.at[index, "min_score"] = min(lista_scores)

**** 0 ****
**** 1 ****
**** 2 ****
**** 3 ****
**** 4 ****
**** 5 ****
**** 6 ****
**** 7 ****
**** 8 ****
**** 9 ****
**** 10 ****
**** 11 ****
**** 12 ****
**** 13 ****
**** 14 ****
**** 15 ****
**** 16 ****
**** 17 ****
**** 18 ****
**** 19 ****
**** 20 ****
**** 21 ****
**** 22 ****
**** 23 ****
**** 24 ****
**** 25 ****
**** 26 ****
**** 27 ****
**** 28 ****
**** 29 ****
**** 30 ****
**** 31 ****
**** 32 ****
**** 33 ****
**** 34 ****
**** 35 ****
**** 36 ****
**** 37 ****
**** 38 ****
**** 39 ****
**** 40 ****
**** 41 ****
**** 42 ****
**** 43 ****
**** 44 ****
**** 45 ****
**** 46 ****
**** 47 ****
**** 48 ****
**** 49 ****
**** 50 ****
**** 51 ****
**** 52 ****
**** 53 ****
**** 54 ****
**** 55 ****
**** 56 ****
**** 57 ****
**** 58 ****
**** 59 ****
**** 60 ****
**** 61 ****
**** 62 ****
**** 63 ****
**** 64 ****
**** 65 ****
**** 66 ****
**** 67 ****
**** 68 ****
**** 69 ****
**** 70 ****
**** 71 ****
**** 72 ****
**** 73 ****
**** 74 ****
**** 75 ****
**** 76 ****
**** 77 *

In [33]:
df_questions_generals['max_score'].astype(float)
df_questions_generals['min_score'].astype(float)
print(df_questions_generals[df_questions_generals["max_score"] >= 0.15].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.20].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.25].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.30].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.35].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.40].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.45].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.50].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.55].shape[0])
print(df_questions_generals[df_questions_generals["max_score"] >= 0.70].shape[0])

406
344
271
201
142
95
56
30
16
0


In [36]:
filtered_generals_quetions = df_questions_generals[df_questions_generals["max_score"] >= 0.20]
filtered_generals_quetions.to_csv('../data/2_processed/questions_generals_reranked.csv', index=False)

In [10]:
df_specific_questions = pd.read_csv('../data/3_gold/eval_retrieval_qwen_kw.csv')
df_specific_questions.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   doc_id             2000 non-null   int64
 1   question           2000 non-null   str  
 2   retrieved_doc_ids  2000 non-null   str  
 3   found              2000 non-null   bool 
dtypes: bool(1), int64(1), str(2)
memory usage: 512.5 KB


In [11]:
df_dataset = pd.read_csv('../data/1_raw/dataset.csv')
df_dataset

,Unnamed: 0,doc_id,abstract,keywords_qwen,kw_v1,kw_v1_df30,question_gemma4_12b,question_gemma4_12b_clean
0,0,0,Wave turbulence is a regime of interacting non...,"turbulence, vortex, wave, kelvin, resonant, ob...","turbulence, vortex, wave, kelvin","turbulence, vortex, wave",What mechanism was identified as driving the e...,What mechanism was identified as driving the e...
1,1,1,Super-resolution estimates a high-resolution i...,"ssim, meteorology, resolution, downscaling, sr...","meteorology, radar, resolution, downscaling","meteorology, radar, resolution, downscaling",What wavelength variations did SRGP-SK reconst...,What wavelength variations did SRGP-SK reconst...
2,2,2,Salt-finger plume forests in a finite layer ca...,"prandtl, diffusivity, roughness, pathway, shea...","diffusivity, roughness, shear, salinity","diffusivity, roughness, shear, salinity",What are the mean absolute differences in w-ac...,What are the mean absolute differences in w-ac...
3,3,3,"Global Navigation Satellite Systems (GNSS), be...","gnss, forecasts, precipitation, prediction, zw...","precipitation, prediction, atmospheric, satellite","precipitation, prediction, atmospheric, satellite",What specific improvement was observed in the ...,What specific improvement was observed in the ...
4,4,4,Deep-learning-based climate downscaling aims t...,"downscaling, adaptation, adaptive, climate, mo...","downscaling, adaptation, adaptive, climate","downscaling, climate",What type of climate projections does the prop...,What type of climate projections does the prop...
...,...,...,...,...,...,...,...,...
8436,8436,8436,A combination of analytical and numerical te...,"convection, circulation, stability, basin, edd...","convection, circulation, stability, basin","convection, circulation, stability, basin",What is the only asymptotic behaviour found in...,What is the only asymptotic behaviour found in...
8437,8437,8437,"In this letter, I solve a model for the dyna...","vortices, vorticity, vortex, kurtosis, enstrop...","vorticity, vortex, diffusion, fluid","vorticity, vortex, diffusion, fluid",What is the scaling law for the number of vort...,What is the scaling law for the number of vort...
8438,8438,8438,The scaling argument developed by Larichev and...,"vorticity, eddy, diffusivity, scaling, stratif...","vorticity, eddy, diffusivity, scale","vorticity, eddy, diffusivity, scale","In the limit x>>1, what are the scaling result...","In the limit x>>1, what are the scaling result..."
8439,8439,8439,The interdecadal variability of a stochastical...,"variability, interdecadal, circulation, ocean,...","variability, stochastic, atmospheric, circulation","variability, stochastic, atmospheric, circulation",What hypothesis does the comparison between th...,What hypothesis does the comparison between th...


In [12]:
resultados = []
for index, row in df_specific_questions.sample(n=500).iterrows():
    query = row['question']
    row_doc = df_dataset.loc[df_dataset['doc_id'] == row['doc_id']]
    abstract = row_doc['abstract'].values[0]

    scores = model.rerank(query, [abstract])
    resultados.append(
        {
            "doc_id": row['doc_id'],
            "question": query,
            "abstract": abstract,
            "score": scores[0]['relevance_score'],
        }
    )

df_resultados = pd.DataFrame(resultados) 

In [16]:
df_resultados['score'].astype(float)
df_resultados[df_resultados['score'] >= 0.70].describe()

,doc_id,score
count,346.000000,346.000000
mean,4241.462428,0.774933
std,2311.401655,0.043843
min,20.000000,0.700205
25%,2120.750000,0.737955
50%,4439.000000,0.771021
75%,6003.000000,0.808161
max,8430.000000,0.900411


In [19]:
df_resultados[df_resultados['score'] >= 0.70].to_csv('../data/2_processed/questions_specific_reranked.csv', index=False)